# 30 requêtes SQL et leur équivalent Pandas

Notebook de l'article [30 requêtes SQL expliquées par leurs équivalents Pandas](https://moncoachdata.com/blog/requetes-sql-expliquees/) de MonCoachData. Base utilisée : [Chinook](https://github.com/lerocha/chinook-database). Code testé avec Python 3.13 et pandas 3.

In [ ]:
!pip install -q duckdb

In [ ]:
import os
from urllib.request import urlretrieve

if not os.path.exists("Chinook_Sqlite.sqlite"):
    urlretrieve(
        "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/requetes-sql-expliquees/Chinook_Sqlite.sqlite",
        "Chinook_Sqlite.sqlite",
    )

## Connexion à la base Chinook

Python embarque SQLite. La fonction `sql()` exécute une requête et renvoie un DataFrame.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("Chinook_Sqlite.sqlite")

def sql(requete):
    """Exécute une requête SQL et renvoie le résultat sous forme de DataFrame."""
    return pd.read_sql_query(requete, conn)

Les 11 tables de la base.

In [ ]:
tables = sql("SELECT name FROM sqlite_master WHERE type = 'table'")

for nom in tables["name"]:
    n = sql(f"SELECT COUNT(*) AS n FROM {nom}").loc[0, "n"]
    print(f"{nom:<14} {n:>5} lignes")

On charge quelques tables dans des DataFrames, pour comparer SQL et Pandas.

In [ ]:
customers = sql("SELECT * FROM Customer")
genres = sql("SELECT * FROM Genre")
tracks = sql("SELECT * FROM Track")
invoices = sql("SELECT * FROM Invoice")
employees = sql("SELECT * FROM Employee")

print(customers.shape, tracks.shape, invoices.shape, employees.shape, genres.shape)

## Premières lignes : SELECT, LIMIT, head()

In [ ]:
sql("""
SELECT * FROM Genre
LIMIT 5
""")

In [ ]:
genres.head()

Sélectionner des colonnes.

In [ ]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
LIMIT 10
""")

In [ ]:
tracks[["Name", "Composer", "UnitPrice"]].head(10)

## DISTINCT et unique()

In [ ]:
sql("""
SELECT DISTINCT FirstName
FROM Employee
""")

In [ ]:
employees["FirstName"].unique().tolist()

## Compter : COUNT

In [ ]:
sql("""
SELECT COUNT(*) FROM Track
""")

In [ ]:
tracks.shape[0]

Compter les valeurs distinctes.

In [ ]:
sql("""
SELECT COUNT(DISTINCT FirstName) FROM Employee
""")

In [ ]:
employees["FirstName"].nunique()

## Filtrer : WHERE et loc[]

In [ ]:
sql("""
SELECT TrackId, Name, UnitPrice
FROM Track
WHERE UnitPrice > 1.0
LIMIT 10
""")

In [ ]:
tracks.loc[tracks["UnitPrice"] > 1.0, ["TrackId", "Name", "UnitPrice"]].head(10)

Plusieurs conditions : AND devient `&` (chaque condition entre parenthèses).

In [ ]:
sql("""
SELECT COUNT(*) FROM Track
WHERE UnitPrice > 1.0 AND UnitPrice < 10
""")

In [ ]:
len(tracks[(tracks["UnitPrice"] > 1) & (tracks["UnitPrice"] < 10)])

OR devient `|`.

In [ ]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity = 'Paris' OR BillingCity = 'Berlin'
LIMIT 5
""")

In [ ]:
villes = (invoices["BillingCity"] == "Paris") | (invoices["BillingCity"] == "Berlin")

invoices.loc[villes, ["BillingAddress", "BillingCity", "Total"]].head(5)

`!=` et `<>` sont équivalents en SQL.

In [ ]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity != 'Paris' AND BillingCity <> 'Berlin'
LIMIT 5
""")

In [ ]:
autres = (invoices["BillingCity"] != "Paris") & (invoices["BillingCity"] != "Berlin")

invoices.loc[autres, ["BillingAddress", "BillingCity", "Total"]].head(5)

## IN et BETWEEN

In [ ]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity IN ('Berlin', 'Paris', 'New York', 'Boston', 'London')
LIMIT 5
""")

In [ ]:
cinq_villes = ["Berlin", "Paris", "New York", "Boston", "London"]

invoices.loc[invoices["BillingCity"].isin(cinq_villes), ["BillingAddress", "BillingCity", "Total"]].head(5)

NOT IN devient `~isin()`.

In [ ]:
sql("""
SELECT BillingAddress, BillingCity, Total
FROM Invoice
WHERE BillingCity NOT IN ('Berlin', 'Paris', 'New York', 'Boston', 'London')
LIMIT 5
""")

In [ ]:
invoices.loc[~invoices["BillingCity"].isin(cinq_villes), ["BillingAddress", "BillingCity", "Total"]].head(5)

BETWEEN inclut les deux bornes.

In [ ]:
sql("""
SELECT BillingCity, Total
FROM Invoice
WHERE Total BETWEEN 5 AND 15
LIMIT 5
""")

In [ ]:
invoices.loc[invoices["Total"].between(5, 15), ["BillingCity", "Total"]].head(5)

## Valeurs manquantes : IS NULL et isnull()

In [ ]:
sql("""
SELECT COUNT(*) FROM Invoice
WHERE BillingState IS NULL
""")

In [ ]:
int(invoices["BillingState"].isnull().sum())

IS NOT NULL devient `notna()`.

In [ ]:
sql("""
SELECT InvoiceDate, BillingCountry
FROM Invoice
WHERE BillingState IS NOT NULL
LIMIT 10
""")

In [ ]:
invoices.loc[invoices["BillingState"].notna(), ["InvoiceDate", "BillingCountry"]].head(10)

## LIKE et les méthodes `str`

In [ ]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Name LIKE 'B%'
LIMIT 5
""")

In [ ]:
tracks.loc[tracks["Name"].str.startswith("B"), ["Name", "Composer", "UnitPrice"]].head(5)

Piège : dans SQLite, LIKE ignore la casse. `str.contains()` la respecte.

In [ ]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Name LIKE '%beautiful%'
""")

In [ ]:
contient = tracks["Name"].str.contains("beautiful")

print("sans case=False :", contient.sum(), "titre(s) trouvé(s)")
print("avec case=False :", tracks["Name"].str.contains("beautiful", case=False).sum(), "titre(s) trouvé(s)")

Avec `case=False`, on obtient le même résultat.

In [ ]:
tracks.loc[tracks["Name"].str.contains("beautiful", case=False), ["Name", "Composer", "UnitPrice"]]

LIKE combiné à une autre condition.

In [ ]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Name LIKE 'F%' AND UnitPrice > 1.0
LIMIT 5
""")

In [ ]:
tracks.loc[
    tracks["Name"].str.startswith("F") & (tracks["UnitPrice"] > 1.0),
    ["Name", "Composer", "UnitPrice"],
].head(5)

## Agrégations : SUM, MAX, MIN, AVG

In [ ]:
sql("""
SELECT SUM(Total), MAX(Total), MIN(Total), AVG(Total)
FROM Invoice
""")

In [ ]:
invoices["Total"].agg(["sum", "max", "min", "mean"])

Piège : une colonne non agrégée sans GROUP BY. SQLite accepte, avec une valeur arbitraire.

In [ ]:
sql("""
SELECT AVG(Total), BillingCity, BillingAddress
FROM Invoice
""")

DuckDB refuse la requête.

In [ ]:
import duckdb

try:
    duckdb.sql("SELECT AVG(Total), BillingCity FROM invoices").df()
except duckdb.Error as erreur:
    print(str(erreur).splitlines()[0])

Agrégation sur un sous-ensemble.

In [ ]:
sql("""
SELECT AVG(Total) AS moyenne
FROM Invoice
WHERE BillingCity = 'Paris'
""")

In [ ]:
float(invoices.loc[invoices["BillingCity"] == "Paris", "Total"].mean())

## Calculs entre colonnes, division entière et alias

In [ ]:
sql("""
SELECT SUM(Total) / COUNT(Total)
FROM Invoice
""")

In [ ]:
float(invoices["Total"].sum() / invoices["Total"].count())

Piège : en SQL, la division de deux entiers donne un entier.

In [ ]:
sql("""
SELECT 10 / 3
""")

In [ ]:
print(10 // 3)    # division entière, comme SQL
print(10 / 3)     # division décimale, comme Python

Avec un nombre décimal, on obtient un résultat décimal.

In [ ]:
sql("""
SELECT 10.0 / 3.0
""")

Durée des titres en minutes.

In [ ]:
sql("""
SELECT Milliseconds / 1000.0 / 60.0
FROM Track
LIMIT 10
""")

In [ ]:
tracks["Milliseconds"].head(10) / 1000 / 60

Alias avec AS. SQLite affiche 2328.6 et Pandas 2328.5999999999995 (arithmétique flottante).

In [ ]:
sql("""
SELECT SUM(Total) AS sum_total
FROM Invoice
""")

In [ ]:
float(invoices["Total"].sum())

Plusieurs agrégats nommés.

In [ ]:
sql("""
SELECT SUM(Total) AS sum_total,
       MAX(Total) AS max_total,
       MIN(Total) AS min_total,
       SUM(Total) / COUNT(Total) AS mean_total
FROM Invoice
""")

In [ ]:
invoices["Total"].agg(sum_total="sum", max_total="max", min_total="min", mean_total="mean")

## Trier : ORDER BY et sort_values()

In [ ]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Composer IS NOT NULL
ORDER BY Composer, TrackId
LIMIT 10
""")

In [ ]:
(tracks[tracks["Composer"].notna()]
    .sort_values(["Composer", "TrackId"])[["Name", "Composer", "UnitPrice"]]
    .head(10))

Piège : SQLite place les NULL en premier, Pandas place les NaN en dernier.

In [ ]:
sql("""
SELECT TrackId, Composer
FROM Track
ORDER BY Composer, TrackId
LIMIT 3
""")

In [ ]:
print(tracks.sort_values(["Composer", "TrackId"])[["TrackId", "Composer"]].head(3))
print()
print(tracks.sort_values(["Composer", "TrackId"], na_position="first")[["TrackId", "Composer"]].head(3))

Tri sur plusieurs colonnes.

In [ ]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Composer IS NOT NULL
ORDER BY UnitPrice, Composer, Name
LIMIT 10
""")

In [ ]:
(tracks[tracks["Composer"].notna()]
    .sort_values(["UnitPrice", "Composer", "Name"])[["Name", "Composer", "UnitPrice"]]
    .head(10))

Tri décroissant (`DESC`) sur certaines colonnes.

In [ ]:
sql("""
SELECT Name, Composer, UnitPrice
FROM Track
WHERE Composer IS NOT NULL
ORDER BY UnitPrice DESC, Composer DESC, Name ASC
LIMIT 10
""")

In [ ]:
(tracks[tracks["Composer"].notna()]
    .sort_values(["UnitPrice", "Composer", "Name"], ascending=[False, False, True])[["Name", "Composer", "UnitPrice"]]
    .head(10))

## Regrouper : GROUP BY et groupby()

In [ ]:
sql("""
SELECT GenreId, COUNT(*) AS nb_titres
FROM Track
GROUP BY GenreId
LIMIT 10
""")

In [ ]:
tracks.groupby("GenreId").size().rename("nb_titres").reset_index().head(10)

Piège : une colonne non groupée. SQLite renvoie un nom quelconque du groupe.

In [ ]:
sql("""
SELECT Name, GenreId, COUNT(*) AS nb_titres
FROM Track
GROUP BY GenreId
LIMIT 5
""")

DuckDB refuse la requête.

In [ ]:
try:
    duckdb.sql("SELECT Name, GenreId, COUNT(*) AS nb_titres FROM tracks GROUP BY GenreId").df()
except duckdb.Error as erreur:
    print(str(erreur).splitlines()[0])

Plusieurs colonnes de groupe et plusieurs agrégats.

In [ ]:
sql("""
SELECT GenreId, AlbumId,
       COUNT(*) AS nb_titres,
       AVG(Milliseconds) / 1000.0 / 60.0 AS duree_moyenne,
       AVG(UnitPrice) AS prix_moyen
FROM Track
GROUP BY GenreId, AlbumId
LIMIT 10
""")

In [ ]:
(tracks.groupby(["GenreId", "AlbumId"])
    .agg(
        nb_titres=("TrackId", "count"),
        duree_moyenne=("Milliseconds", lambda ms: ms.mean() / 1000 / 60),
        prix_moyen=("UnitPrice", "mean"),
    )
    .reset_index()
    .head(10))

Tri des groupes sur le résultat d'un agrégat.

In [ ]:
sql("""
SELECT GenreId, AlbumId,
       COUNT(*) AS nb_titres,
       AVG(Milliseconds) / 1000.0 / 60.0 AS duree_moyenne,
       AVG(UnitPrice) AS prix_moyen
FROM Track
GROUP BY GenreId, AlbumId
ORDER BY duree_moyenne DESC, nb_titres DESC
LIMIT 10
""")

In [ ]:
(tracks.groupby(["GenreId", "AlbumId"])
    .agg(
        nb_titres=("TrackId", "count"),
        duree_moyenne=("Milliseconds", lambda ms: ms.mean() / 1000 / 60),
        prix_moyen=("UnitPrice", "mean"),
    )
    .reset_index()
    .sort_values(["duree_moyenne", "nb_titres"], ascending=False)
    .head(10))

## Filtrer les groupes : HAVING

WHERE ne peut pas filtrer un agrégat.

In [ ]:
import sqlite3

requetes = [
    "SELECT GenreId FROM Track GROUP BY GenreId WHERE COUNT(GenreId) > 100",
    "SELECT GenreId FROM Track WHERE COUNT(GenreId) > 100 GROUP BY GenreId",
]

for requete in requetes:
    try:
        conn.execute(requete)
    except sqlite3.Error as erreur:
        print(f"{type(erreur).__name__} : {erreur}")

La clause qui filtre les groupes est HAVING.

In [ ]:
sql("""
SELECT GenreId, COUNT(*) AS nb_titres
FROM Track
GROUP BY GenreId
HAVING COUNT(*) > 100
""")

In [ ]:
nb_titres = tracks.groupby("GenreId").size().rename("nb_titres")

nb_titres[nb_titres > 100].reset_index()

## Joindre deux tables : JOIN et merge()

In [ ]:
sql("""
SELECT Track.Name AS titre, Genre.Name AS genre
FROM Track
JOIN Genre ON Track.GenreId = Genre.GenreId
LIMIT 5
""")

In [ ]:
(tracks.merge(genres, on="GenreId", suffixes=("_titre", "_genre"))
    [["Name_titre", "Name_genre"]]
    .rename(columns={"Name_titre": "titre", "Name_genre": "genre"})
    .head(5))

## SQL directement sur un DataFrame avec DuckDB

In [ ]:
duckdb.sql("""
    SELECT GenreId, COUNT(*) AS nb_titres
    FROM tracks
    GROUP BY GenreId
    HAVING COUNT(*) > 100
    ORDER BY GenreId
""").df()